# Golden set: label the pool, then score the routes

A **golden set** is a small set of target-comp pairs scored by hand. The pool to label is **every comp any route found** for the two golden-set targets, **PCTY and WEN** (chosen in section 9 of `comps_and_valuation.ipynb`). Pooling means a route is judged on the peers the other routes found too, not only on its own shortlist.

The notebook has three steps:

1. **Run down to section 2.** It builds the pool and writes the sheet to label, `outputs-local/comps_pilot_targets/golden_set/labels/golden_set_labels.csv`. It never overwrites that file once it exists.
2. **Fill in the sheet** (Excel is fine; keep it a CSV). Rank, similarity, the LLM's verdicts and EV/EBITDA are not on it, and the rows are shuffled.
3. **Re-run from section 3.** It checks the sheet and, once every pair is reviewed, prints the metrics.

Nothing here calls an API or re-embeds anything; it reads the saved outputs of the main notebook.

### How to score

| score | meaning |
|---|---|
| **2** | direct competitor: same product, same customers |
| **1** | adjacent: overlapping product or customers |
| blank | **0**: anything else. Every 0 gets **one** reason in `reason_if_0` |

Reasons for a 0 (pick one of these five, spelled as shown):

| reason | example |
|---|---|
| Different business model | Barrett Business Services for Paylocity: same customers, but a PEO |
| Different customers | Enterprise software matched to an SMB product |
| Different scale | A $50M company matched to a $20B one |
| Generic wording only | Matched on "platform", "solutions", "clients" |
| Wrong industry code | SIC groups unlike businesses (Prologis and Park Hotels share a REIT code) |

Because a blank score means 0, a blank score **with no reason** is treated as "not reviewed yet", and the metrics wait until none are left. Use the `notes` column for anything else. The sheet also shows each company's website and the model-written offering and customer text (from the facets), only to save you a lookup; score from what the companies do.

### Metrics (k = 10)

| metric | meaning |
|---|---|
| **Precision@10** | share of a route's top 10 scored 1 or 2 |
| **Direct hits@10** | number of direct competitors (2s) in its top 10 |
| **NDCG@10** | whether the best peers are ranked near the top, 0 to 1 |

NDCG in one line: each peer in the top 10 earns its score (0, 1 or 2), discounted the lower it sits (divided by log2 of its position plus 1); the sum is divided by the same sum for the best possible order of that target's whole labeled pool. 1.0 means the route's top 10 is the best order the pool allows; a route that puts the 2s first beats one that puts them tenth even at equal precision. A route that lists fewer than 10 is scored on what it listed for precision, but NDCG still counts the best peers it left out against it.

Limits: peers that no route found are not in the pool, so these measure how well each route ranks and filters what was found, not recall. Two targets are a calibration sample, not an accuracy estimate.

In [1]:
import json
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

# repo root = the folder holding data_manifest.json (works from the repo root, v0/ or v0/notebooks/)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_manifest.json").exists())
sys.path.insert(0, str(ROOT / "v0"))
import facets as fc                     # also puts website_scrape/ and src/ on the path
import golden_set as gs
from fill_facets import DEFAULT_OUT as FACETS_OUT, output_paths
from project_data import OUTPUTS_LOCAL

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 200)

RUN = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
COMPS_OUT = OUTPUTS_LOCAL / "comps"
BASE = OUTPUTS_LOCAL / "comps_pilot_targets"          # the main notebook's outputs, read here
GOLDEN_ROOT = BASE / "golden_set"
LABELS_DIR = GOLDEN_ROOT / "labels"                   # edited by hand: stable names, never overwritten
LABELS = LABELS_DIR / "golden_set_labels.csv"
POOL_FILE = LABELS_DIR / "pool_with_routes.csv"       # which route picked each pair, at what rank (not for the labeler)
GOLDEN = ["PCTY", "WEN"]                              # section 9 of comps_and_valuation.ipynb
K = gs.K

def latest(folder, name):
    return sorted(p for p in folder.iterdir() if (p / name).exists())[-1]

UNIVERSE_RUN = latest(COMPS_OUT / "universe", "universe.csv")
CAND_RUN = latest(BASE / "candidates", "ranking.csv.gz")
LTC_RUN = latest(BASE / "llm_then_cosine", "pairs.csv")
universe = pd.read_csv(UNIVERSE_RUN / "universe.csv", dtype=str).set_index("IQ_COMPANY_ID")
id_of = {t: i for i, t in universe["target_ciq_ticker"].items()}
ticker = universe["target_ciq_ticker"]
GOLDEN_IDS = [id_of[t] for t in GOLDEN]
print(f"universe {UNIVERSE_RUN.name}, candidates {CAND_RUN.name}, LLM then cosine {LTC_RUN.name}; golden-set targets: {GOLDEN}")

universe 20261025T060358Z, candidates 20261027T204930Z, LLM then cosine 20261027T204930Z; golden-set targets: ['PCTY', 'WEN']


## 1. The routes and the pool to label

Four routes, each a ranked list of comps per target (rank 1 = best), all from the main notebook's saved outputs:

| route | list |
|---|---|
| cosine similarity | the step 4 candidates (top 10 or z ≥ 2), most similar first |
| cosine then LLM good comps | the same candidates the LLM kept (score 1), in the same order |
| LLM picks directly | the LLM's own picks from the whole training pool (up to 10) |
| LLM picks then cosine | those direct picks, reordered by cosine similarity |

The pool is every (target, comp) pair in any of them.

In [2]:
ranking = pd.read_csv(CAND_RUN / "ranking.csv.gz")
cosine = ranking[ranking["z_ge_2"] | ranking["in_top_k"]][["target", "comp", "rank"]]

judged = pd.read_csv(BASE / "judge" / "tables" / "judgments.csv")
good = judged[judged["score"] == 1][["target", "comp", "rank"]].copy()
good["rank"] = good.groupby("target")["rank"].rank(method="first").astype(int)   # position among the kept comps

picks = pd.read_csv(BASE / "suggest" / "tables" / "suggestions.csv").dropna(subset=["comp"])[["target", "comp", "rank"]]
picks["rank"] = picks["rank"].astype(int)

then_cosine = pd.read_csv(LTC_RUN / "pairs.csv")[["target", "comp", "rank"]]

ROUTES = {"cosine similarity": cosine,
          "cosine then LLM good comps": good,
          "LLM picks directly": picks,
          "LLM picks then cosine": then_cosine}
for name, df in ROUTES.items():
    assert not df.duplicated(["target", "comp"]).any(), name

pool = gs.build_pool(ROUTES, GOLDEN_IDS)
rank_cols = [c for c in pool.columns if c.startswith("rank: ")]
counts = pd.DataFrame({name: df[df["target"].isin(GOLDEN_IDS)].groupby("target").size() for name, df in ROUTES.items()})
counts.index = counts.index.map(ticker)
counts["pool to label"] = pool.groupby("target_id").size().rename(index=ticker)
print("pairs per route and the pool to label:")
display(counts)

pairs per route and the pool to label:


,cosine similarity,cosine then LLM good comps,LLM picks directly,LLM picks then cosine,pool to label
target,,,,,
WEN,20,4,3,3,20
PCTY,15,4,5,5,16


## 2. The sheet to label

One row per pair, target on the left and comp on the right, in a shuffled order. Fill in `score` (2 or 1; leave blank for 0) and, for every 0, `reason_if_0`. If the sheet already exists it is kept as it is, and only checked against the pool.

In [3]:
facets = pd.read_csv(output_paths(FACETS_OUT).tables / "facets.csv", dtype=str).fillna("")
facets = facets[(facets["prompt_version"] == fc.PROMPT_VERSION) & (facets["llm_status"] == "ok") & (facets["business_id"] == "B1")]
facets = facets.set_index("IQ_COMPANY_ID")
shown_all = pd.DataFrame({
    "ticker": universe["target_ciq_ticker"],
    "company": universe["target_company_name"],
    "sic_industry": universe["target_primary_sic_industry"],
    "website": universe["target_ciq_website"],
    "offering": facets["offering_description"].reindex(universe.index).fillna(""),
    "customers": facets["customer_segment"].reindex(universe.index).fillna(""),
})
sheet = gs.labeler_sheet(pool, {"target": shown_all, "comp": shown_all}, seed=2026)

LABELS_DIR.mkdir(parents=True, exist_ok=True)
if LABELS.exists():
    print(f"kept the existing sheet: {LABELS.relative_to(ROOT)} (not overwritten)")
else:
    sheet.to_csv(LABELS, index=False, encoding="utf-8-sig")        # the BOM lets Excel read the accents
    print(f"wrote the blank sheet: {LABELS.relative_to(ROOT)} ({len(sheet)} pairs)")

# which route found which pair, with ranks: kept beside the sheet, not shown to the labeler
route_view = pool.assign(target=pool["target_id"].map(ticker), comp=pool["comp_id"].map(ticker),
                         comp_company=pool["comp_id"].map(universe["target_company_name"]))
route_view = route_view[["target", "comp", "comp_company", *rank_cols, "target_id", "comp_id"]]
if POOL_FILE.exists():
    old = pd.read_csv(POOL_FILE)
    if not old.equals(route_view.reset_index(drop=True).astype(old.dtypes.to_dict())):
        history = GOLDEN_ROOT / "history"
        history.mkdir(exist_ok=True)
        shutil.move(str(POOL_FILE), history / f"pool_with_routes_{RUN}.csv")      # a replaced file is kept, not overwritten
        print("the pool changed since the last run: the old pool_with_routes.csv moved to golden_set/history/")
if not POOL_FILE.exists():
    route_view.to_csv(POOL_FILE, index=False)
    print("wrote", POOL_FILE.relative_to(ROOT))
sheet[["pair", "target_ticker", "comp_ticker", "comp_company", "comp_sic_industry"]].head(8)

wrote the blank sheet: outputs-local\comps_pilot_targets\golden_set\labels\golden_set_labels.csv (36 pairs)
wrote outputs-local\comps_pilot_targets\golden_set\labels\pool_with_routes.csv


,pair,target_ticker,comp_ticker,comp_company,comp_sic_industry
0,WEN-01,WEN,SENE.A,Seneca Foods Corporation,"Canned Fruits, Vegetables, Preserves, Jams, and Jellies"
1,WEN-02,WEN,VNCE,Vince Holding Corp.,Apparel and Other Finished Products Made From Fabrics and Similar ...
2,WEN-03,WEN,FRPT,"Freshpet, Inc.",Grain Mill Products
3,WEN-04,WEN,BGS,"B&G Foods, Inc.",Food and Kindred Products
4,WEN-05,WEN,BDL,"Flanigan's Enterprises, Inc.",Eating Places
5,WEN-06,WEN,STKL,SunOpta Inc.,Miscellaneous Food Preparations And Kindred Products
6,WEN-07,WEN,GIII,"G-III Apparel Group, Ltd.",Apparel and Other Finished Products Made From Fabrics and Similar ...
7,WEN-08,WEN,TR,"Tootsie Roll Industries, Inc.",Sugar And Confectionery Products


## 3. Check the labels

Reads the sheet back. Everything must be consistent before the metrics are printed: no pair missing or listed twice, every score blank, 0, 1 or 2, every 0 with exactly one of the five reasons, and no reason on a 1 or 2. A blank score with no reason is a pair not reviewed yet.

In [4]:
labels = gs.read_labels(LABELS)
problems = gs.check_labels(labels, pool)
READY = problems.empty
if READY:
    final = gs.scores(labels)
    print(f"{len(final)} pairs reviewed, all consistent.")
    by_target = pd.crosstab(final["pair"].str.split("-").str[0], final["score"].map({2: "2 direct", 1: "1 adjacent", 0: "0"}))
    display(by_target)
    zeros = final[final["score"] == 0]
    display(pd.crosstab(zeros["pair"].str.split("-").str[0], zeros["reason"]))
else:
    unreviewed = problems["problem"].str.startswith("not reviewed").sum()
    print(f"labels not ready: {len(problems)} problems ({unreviewed} pairs not reviewed yet, of {len(pool)}).")
    print(f"fill in {LABELS.relative_to(ROOT)}, save it as a CSV, then re-run from here.")
    display(problems.head(40))

labels not ready: 36 problems (36 pairs not reviewed yet, of 36).
fill in outputs-local\comps_pilot_targets\golden_set\labels\golden_set_labels.csv, save it as a CSV, then re-run from here.


,pair,problem
0,WEN-01,not reviewed: a 0 (blank score) needs one reason
1,WEN-02,not reviewed: a 0 (blank score) needs one reason
2,WEN-03,not reviewed: a 0 (blank score) needs one reason
3,WEN-04,not reviewed: a 0 (blank score) needs one reason
4,WEN-05,not reviewed: a 0 (blank score) needs one reason
5,WEN-06,not reviewed: a 0 (blank score) needs one reason
6,WEN-07,not reviewed: a 0 (blank score) needs one reason
7,WEN-08,not reviewed: a 0 (blank score) needs one reason
8,WEN-09,not reviewed: a 0 (blank score) needs one reason
9,WEN-10,not reviewed: a 0 (blank score) needs one reason


## 4. Metrics

For each route and target, over the route's top 10: how many it listed, precision@10, direct hits@10 and NDCG@10, then the mean over the two targets. Scores are the hand scores above; a pair scored 0 stays in the denominator.

In [5]:
if READY:
    final = gs.scores(labels)
    metrics = gs.evaluate(ROUTES, final, GOLDEN_IDS, k=K)
    metrics["target"] = metrics["target"].map(lambda t: ticker.get(t, t))
    out = GOLDEN_ROOT / "metrics" / RUN
    out.mkdir(parents=True, exist_ok=True)
    metrics.to_csv(out / "metrics.csv", index=False)
    final.assign(target=final["target_id"].map(ticker), comp=final["comp_id"].map(ticker)).to_csv(out / "scores.csv", index=False)
    (out / "run.json").write_text(json.dumps({
        "run": RUN, "golden_targets": GOLDEN, "k": K, "pairs": len(final), "candidates_run": CAND_RUN.name,
        "llm_then_cosine_run": LTC_RUN.name, "labels_file": str(LABELS.relative_to(ROOT)),
        "routes": list(ROUTES)}, indent=1), encoding="utf-8")
    print("wrote", out.relative_to(ROOT))
    display(metrics[metrics["target"] == "mean"].drop(columns="target").set_index("route").round(2))
    display(metrics[metrics["target"] != "mean"].set_index(["target", "route"]).round(2))
else:
    print("waiting for the labels (section 3).")

waiting for the labels (section 3).


**How to read it.** Precision@10 says how clean a route's list is; direct hits@10 says how many real competitors it surfaced (the routes list different numbers, so compare it with the "listed" column); NDCG@10 adds whether the best ones sit on top and whether the route missed peers another route found. With two targets and 16-20 pairs each, one pair moves a number a lot: read the table, not the third decimal. The LLM judge's labels (`judgments.csv`) can be compared to these scores afterwards to measure its precision and recall; that comparison is not built here.